In [ ]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv("Gold price.csv")

In [ ]:
df.head()

In [ ]:
df["Gold_rate"]=df["Gold_rate"].replace("₹","",regex=True).replace(",","",regex=True).astype("float")

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
sns.boxplot(x="USD_INR",data=df)

In [ ]:
df["USD_INR"].max()

In [ ]:
sns.regplot(x="USD_INR",y="Gold_rate",data=df)

In [ ]:
x=df[["USD_INR"]]
y=df[["Gold_rate"]]

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.1,random_state=2)

In [ ]:
x.shape,x_train.shape,x_test.shape

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

In [ ]:
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
regressor=LinearRegression()

In [ ]:
regressor.fit(x_train_scaled,y_train)

In [ ]:
regressor.get_params()

In [ ]:
regressor.coef_

In [ ]:
regressor.intercept_

In [ ]:
x_train_predict=regressor.predict(x_train_scaled)

In [ ]:
plt.scatter(x_train,y_train)
plt.plot(x_train,x_train_predict,color="r")
plt.xlabel("USD_INR")
plt.ylabel("Goldrate")
plt.show()

In [ ]:
x_test_predicted=regressor.predict(x_test_scaled)

In [ ]:
x_test_predicted

In [ ]:
y_test

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
mean_squared_error(y_test,x_test_predicted)

# Hyperparameter Optimization

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
para_space={"copy_X":[True,False],"fit_intercept":[True,False],"n_jobs":[1,5,10,15,None],"positive":[True,False]}

In [ ]:
search=RandomizedSearchCV(regressor,para_space,n_iter=50,cv=5)

In [ ]:
search.fit(x_train,y_train)

In [ ]:
search.best_params_

In [ ]:
tuned_model=LinearRegression(positive= True, n_jobs= 1, fit_intercept= True, copy_X= True)

In [ ]:
tuned_model.fit(x_train_scaled,y_train)

In [ ]:
y_pred=tuned_model.predict(x_test_scaled)

In [ ]:
from sklearn.metrics import r2_score
print("R2 Score:", r2_score(y_test, y_pred))

In [ ]:
def calculate_gold_rate(usd_inr):
    scaled_input=scaler.transform(np.array(usd_inr).reshape(1,-1))
    return regressor.predict(scaled_input)[0][0].round(2)
    

In [ ]:
calculate_gold_rate(92)

In [ ]:
import gradio as gr

def greet(name, intensity):
    return "Hello, " + name + "!" * int(intensity)

demo = gr.Interface(
    fn=calculate_gold_rate,
    inputs=["number"],
    outputs=["number"],
    title="How much is 1gm gold"
)

demo.launch()
